In [ ]:
from src.config import DATA_ROOT, FIGURE_ROOT, SEQ_LENGTH
from pathlib import Path
for folder in ["", "shap", "embeddings"]:
    (Path(FIGURE_ROOT) / folder).mkdir(parents=True, exist_ok=True)


# AUC ROC Statistical Test

In [ ]:
import pickle
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, average_precision_score

def eval_res(y_preds, y_trues):
    pred_class = np.array(y_preds) > 0.5
    scores = {
        "confmats": confusion_matrix(y_trues, pred_class, labels=[0, 1]),
        "f1": f1_score(y_trues, pred_class, average='binary'),
        "pr_auc": average_precision_score(y_trues, y_preds),
    }
    try:
        scores["roc_auc"] = roc_auc_score(y_trues, y_preds)
    except:
        scores["roc_auc"] = 0.0
    
    # 
    cf = scores["confmats"]
    TN, TP, FP, FN = cf[0][0], cf[1][1], cf[0][1], cf[1][0]
    scores["specificity"] = TN / (TN + FP)
    scores["sensitivity"] = TP / (TP + FN)
    scores["balance_acc"] = (scores["specificity"]+scores["sensitivity"]) / 2
    return scores

def eval_res_multi(y_preds, y_trues, verbose=True):
    scores = dict()
    for task in y_preds:
        scores[task] = eval_res(y_preds[task], y_trues[task])

        if verbose:
            print(task)
            for s in scores[task]:
                print(s, scores[task][s])
            print("======\n")
    return scores

def calc_res(trail_name, verbose=True):
    # Historical single-task summary variant: requires flat prediction/target arrays and the matching accumulation/reporting blocks.
    # # single task
    # scores = {
    #     "confmats": list(),
    #     "f1": list(),
    #     "pr_auc": list(),
    #     "roc_auc": list(),
    #     "specificity": list(),
    #     "sensitivity": list(),
    #     "balance_acc": list()
    # }

    # multitask
    scores = dict()

    for i in tqdm([
        0,
        1,
        2,
        3,
        4
    ]):  
        record_path = (DATA_ROOT + '/exp_res/{}/{}_record_{}').format(trail_name, trail_name, i)
        with open(record_path, "rb") as f:
            record = pickle.load(f)
        
        # Single-task variant: paired with the flat score dictionary and reporting block.
        # single task
        # curr_scores = eval_res(record["y_preds"], record["y_trues"])
        # for s in curr_scores:
        #     scores[s].append(curr_scores[s])

        # add the criteria with "and"
        post_task = {
            'nadir90fall20': {
                "y_preds": [record["y_preds"]["nadir90"][j]*record["y_preds"]["fall20"][j] for j in range(len(record["y_preds"]["nadir90"]))],
                "y_trues": [record["y_trues"]["nadir90"][j] and record["y_trues"]["fall20"][j] for j in range(len(record["y_preds"]["nadir90"]))]
            },
            'nadir90fall30': {
                "y_preds": [record["y_preds"]["nadir90"][j]*record["y_preds"]["fall30"][j] for j in range(len(record["y_preds"]["nadir90"]))],
                "y_trues": [record["y_trues"]["nadir90"][j] and record["y_trues"]["fall30"][j] for j in range(len(record["y_preds"]["nadir90"]))]
            },
            'fall20kdoqi': {
                "y_preds": [record["y_preds"]["kdoqi"][j]*record["y_preds"]["fall20"][j] for j in range(len(record["y_preds"]["kdoqi"]))],
                "y_trues": [record["y_trues"]["kdoqi"][j] and record["y_trues"]["fall20"][j] for j in range(len(record["y_preds"]["kdoqi"]))]
            }
        }
        for t in post_task:
            record["y_preds"][t] = post_task[t]['y_preds']
            record["y_trues"][t] = post_task[t]['y_trues']

        # multitask
        curr_scores = eval_res_multi(record["y_preds"], record["y_trues"], verbose=False)
        if len(scores) == 0:
            for task in curr_scores:
                scores[task] = {
                    "confmats": list(),
                    "f1": list(),
                    "pr_auc": list(),
                    "roc_auc": list(),
                    "specificity": list(),
                    "sensitivity": list(),
                    "balance_acc": list()
                }
        for task in curr_scores:
            if task == "fall20" and verbose:
                print(curr_scores[task]['f1'])
            for s in curr_scores[task]:
                scores[task][s].append(curr_scores[task][s])
    
    # Single-task variant: report the flat score dictionary instead of task-keyed metrics.
    # # single task
    # for s in scores:
    #     if s == "confmats":
    #         print(s)
    #         print(np.sum(scores[s], axis=0))
    #         continue
    #     print(s, np.mean(scores[s]), "+-", np.std(scores[s]))
    
    # multitask
    if verbose:
        for task in scores:
            print(task)
            for s in scores[task]:
                if s == "confmats":
                    print(s)
                    print(np.sum(scores[task][s], axis=0))
                    continue
                print(s, round(np.mean(scores[task][s]), 3), "+-", round(np.std(scores[task][s]), 3))
            print("======\n")
    else:
        return scores


In [ ]:
scores = calc_res('pre_real_all_raw_ehr', verbose=False)
scores.keys()


In [ ]:
no_ehr_scores = calc_res('pre_real_base_multi_early', verbose=False)
no_ehr_scores.keys()


In [ ]:
late_fuse_scores = calc_res('pre_real_base_multi', verbose=False)
late_fuse_scores.keys()


In [ ]:
lstm_scores = calc_res('baseline', verbose=False)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as ss
import scikit_posthocs as sp

def cd_test(dict_data):


    # ranking
    data = (
    pd.DataFrame(dict_data)
    .rename_axis('cv_fold')
    .melt(
        var_name='estimator',
        value_name='score',
        ignore_index=False,
    )
    .reset_index()
    )

    avg_rank = data.groupby('cv_fold').score.rank(pct=True).groupby(data.estimator).mean()

    plt.clf()
    plt.figure(figsize=(8, 6), dpi=100)
    plt.rc('font', size=16)
    cmap_custom = [
            "#FFFFFF",  # 对角线 — 白色
            "#E0ECF5",  # 非显著 — 浅蓝灰
            "#7EBADD",  # p < 0.001 — 粉蓝深
            "#A2C8E6",  # p < 0.01 — 粉蓝中等 (主色调)
            "#C6DBEF",  # p < 0.05 — 粉蓝浅
        ]
    test_results = sp.posthoc_conover_friedman(
        data,
        melted=True,
        block_col='cv_fold',
        block_id_col='cv_fold',
        group_col='estimator',
        y_col='score',
    )
    print(test_results)
    ax, cbar = sp.sign_plot(
        test_results,
        flat=False,
        cmap=cmap_custom,
        labels=True 
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
    plt.show()

    plt.clf()
    plt.figure(figsize=(10, 2), dpi=100)
    sp.critical_difference_diagram(
        avg_rank, 
        test_results,
        crossbar_props={'color': None, 'marker': 'o'}
    )
    plt.show()


In [ ]:
import scipy.stats as st
from scipy.stats import permutation_test

def statistic(x, y):
    return np.mean(x) - np.mean(y)

def ci(lst, confidence=0.95):
    return st.t.interval(confidence=confidence, df=len(lst)-1, 
            loc=np.mean(lst), scale=st.sem(lst))

def ci_and_stat_test(scores, score_key='roc_auc'):
    # score_key: roc_auc or balance_acc
    # extract list of scores from all IDH def
    score_keys = [
        'roc_auc', 
        # 'balance_acc', 
        # 'sensitivity', 
        # 'specificity'
    ]

    scores_list = dict()
    for m_i in range(len(scores)):
        scores_list[m_i] = list()
        for idh_def in scores[m_i]:
            for score_key in score_keys:
                scores_list[m_i] += scores[m_i][idh_def][score_key]
        print("Method {} 95% CI:".format(m_i+1), ci(scores_list[m_i], confidence=0.95))
        print("Method {}:".format(m_i+1), np.mean(scores_list[m_i]), np.std(scores_list[m_i]), len(scores_list[m_i]))

    # Statistical-test alternatives: permutation test or independent t-test instead of the active comparison.
    # conduct permutation test whether scores1 is larger than scores2
    # stat_res = permutation_test((
    #     np.array(scores1_list), 
    #     np.array(scores2_list)
    # ), statistic, alternative='greater', n_resamples=10000, random_state=42)
    # p_value = stat_res.pvalue
    # t_statistic, p_value = st.ttest_ind(scores1_list, scores2_list, alternative='greater')
    # print("Statistical Test Result:", p_value)

    dict_data = {
        'Final': np.array(scores_list[0]),
        'w/o Care Records': np.array(scores_list[1]),
        'Late Fusion': np.array(scores_list[2]),
        'Baseline': np.array(scores_list[3]),
    }
    cd_test(dict_data)

# Comparison variant: full model versus the model without EHR.
# ci_and_stat_test(scores, no_ehr_scores, score_key='roc_auc')
ci_and_stat_test(
    [
        scores, 
        no_ehr_scores, 
        late_fuse_scores,
        lstm_scores
    ], 
    score_key='roc_auc'
)
# Comparison variant: model without EHR versus the baseline.
# ci_and_stat_test(no_ehr_scores, lstm_scores, score_key='roc_auc')


In [ ]:
# compare if using mean and std
rng = np.random.default_rng()
t_statistic, p_value = st.ttest_ind(
    rng.normal(loc=0.82035, scale=0.088, size=100000), 
    rng.normal(loc=0.8176, scale=0.090, size=100000), 
    # Statistical-test variant: Welch's t-test instead of assuming equal variance.
    # equal_var=False
    alternative='greater'
)
p_value


In [ ]:
# Test Late Fuse and no multitask
permutation_test((
        [0.84, 0.86, 0.94, 0.93, 0.81, 0.80, 0.91, 0.91, 0.91], 
        [0.71, 0.74, 0.88, 0.86, 0.81, 0.81, 0.85, 0.83, 0.90]
    ), statistic, alternative='greater', n_resamples=10000, random_state=42).pvalue


In [ ]:
# Preliminary results on fuse record or not
rng = np.random.default_rng()
permutation_test((
        rng.normal(loc=0.812, scale=0.011, size=100), 
        rng.normal(loc=0.804, scale=0.011, size=100)
    ), statistic, alternative='greater', n_resamples=10000, random_state=42).pvalue


# Some Statistics

In [ ]:
with open((DATA_ROOT + '/splits_5fold_all'), "rb") as f:
    splits = pickle.load(f)

for split in splits:
    print("# Train:", len(split['train_fnames']))
    print("# Test:", len(split['test_fnames']))
    print("\n")


In [ ]:
import pickle

m_name = 'pre_real_all_adjust_ehr'

stats = dict() # map: (has_drop, has_symp, has_interv) -> num
total = 0

for fold_i in range(5):
    # fetch file names
    with open((DATA_ROOT + '/splits_5fold_all'), "rb") as f:
        split = pickle.load(f)[fold_i]

    # fetch output
    with open((DATA_ROOT + '/exp_res/{}/{}_record_{}').format(m_name, m_name, fold_i), "rb") as f:
        records = pickle.load(f) 
    
    # iterate over labels
    for i in range(len(records['y_trues']['fall20'])):
        has_drop = records['y_trues']['fall20'][i] or records['y_trues']['nadir90'][i]
        has_symp = records['y_trues']['hemo'][i]
        has_interv = records['y_trues']['kdoqi'][i]

        curr_key = (has_drop, has_symp, has_interv)

        if stats.get(curr_key) is None:
            stats[curr_key] = 0
        
        stats[curr_key] += 1
        total += 1

for k in stats:
    print(k, stats[k], stats[k] / (total-stats[(0, 0, 0)]))


# 3. Performance over time

In [ ]:
import pickle
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, average_precision_score

def eval_score(y_trues, y_preds, metric='auroc'):
    thres = 0.5
    tn, fp, fn, tp = confusion_matrix(y_trues, (np.array(y_preds) >= thres).astype(int)).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    if metric == 'auroc':
        return roc_auc_score(y_trues, y_preds)
    elif metric == 'spec':
        return specificity
    elif metric == 'sens':
        return sensitivity
    elif metric == 'bal_acc':
        return 0.5*(specificity+sensitivity)

def score_over_time(m_name='pre_real_all_adjust_ehr', fold_i=0, chosen_def='nadir90', plot=True, metric='auroc'):
    # fetch file names
    with open((DATA_ROOT + '/splits_5fold_all'), "rb") as f:
        split = pickle.load(f)[fold_i]


    # fetch output
    with open((DATA_ROOT + '/exp_res/{}/{}_record_{}').format(m_name, m_name, fold_i), "rb") as f:
        records = pickle.load(f) # 

    len(records["y_preds"][chosen_def])

    # re-fetch index
    time_step_idxs = dict() # map: num_observe_available -> list(idxs)
    for f_i in range(len(split["test_fnames"])):
        pid, sid, sidx = split["test_fnames"][f_i].split('_')
        sidx = int(sidx)
        if time_step_idxs.get(sidx) is None:
            time_step_idxs[sidx] = list()
        time_step_idxs[sidx].append(f_i)

    # calculate performance for each time step group
    y_preds = np.array(records["y_preds"][chosen_def])
    y_trues = np.array(records["y_trues"][chosen_def])

    scores_steps = dict()
    for num_observe in time_step_idxs:
        curr_idxs = time_step_idxs[num_observe]
        # Analysis variant: exclude time bins containing fewer than ten observations.
        # if len(curr_idxs) < 10:
        #     continue
        if num_observe == 0 or num_observe >= 12:
            continue
        scores_steps[num_observe] = eval_score(y_trues[curr_idxs], y_preds[curr_idxs], metric=metric)

    scores_steps = sorted(scores_steps.items(), key=lambda x: x[0])
    step_keys = [s[0] for s in scores_steps]
    scores_steps = [s[1] for s in scores_steps]

    if plot:
        plt.plot(step_keys, scores_steps, label=f"Fold {fold_i}", marker='o')
        plt.ylim(0.5, 1.0)
    else:
        return step_keys, scores_steps

# function call
plt.clf()
for i in range(5):
    score_over_time(m_name='pre_real_all_adjust_ehr', fold_i=i, chosen_def='nadir90')
plt.legend()
plt.show()


In [ ]:
plt.clf()
plt.figure(figsize=(6,4))

# config
chosen_def = 'hemo'
metric_names = {
    'auroc': 'AUC ROC',
    'spec': 'Specificity',
    'sens': 'Sensitivity',
    'bal_acc': 'Balanced Accuracy'
}
colors = {
    'auroc': "#F6A5B4",
    'spec': "#A2C8E6",
    'sens': "#A3E4D7",
    'bal_acc': "#FF6B6B"
}

# fetch auc roc score
for metric in tqdm(metric_names):
    scores = list()
    for i in range(5):
        step_keys, scores_steps = score_over_time(m_name='pre_real_all_adjust_ehr', fold_i=i, chosen_def=chosen_def, plot=False, metric=metric)
        scores.append(scores_steps)
    scores = np.array(scores)
    mean, std = scores.mean(axis=0), scores.std(axis=0)
    plt.errorbar(np.arange(len(mean)), mean, yerr=std, fmt='-o', capsize=3, color=colors[metric], lw=2, markersize=5, label=metric_names[metric])

# figure config
plt.xticks(np.arange(len(mean))+1)
plt.ylabel('AUC ROC')
plt.xlabel('Number of Observed Measurements')
plt.title(chosen_def[0].upper()+chosen_def[1:] if chosen_def != 'hemo' else "HEMO")
plt.ylim(0.09, 1.01) # for hemo
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# ------------------------
# Global matplotlib config (journal style)
# ------------------------
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "lines.linewidth": 2.5,
    "lines.markersize": 6,
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ------------------------
# Config
# ------------------------
chosen_defs = ['fall20', 'nadir90', 'hemo']

metric_names = {
    'auroc': 'AUC ROC',
    'spec': 'Specificity',
    'sens': 'Sensitivity',
    'bal_acc': 'Balanced Accuracy'
}

colors = {
    'auroc': "#F6A5B4",
    'spec': "#4C72B0",
    'sens': "#55A868",
    'bal_acc': "#C44E52"
}

# ------------------------
# Create figure with 3 panels
# ------------------------
fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(18, 5),
    sharey=False
)

# ------------------------
# Plot each definition
# ------------------------
for ax, chosen_def in zip(axes, chosen_defs):

    for metric in metric_names:
        scores = []
        for i in range(5):
            step_keys, scores_steps = score_over_time(
                m_name='pre_real_all_adjust_ehr',
                fold_i=i,
                chosen_def=chosen_def,
                plot=False,
                metric=metric
            )
            scores.append(scores_steps)

        scores = np.array(scores)
        mean = scores.mean(axis=0)
        std = scores.std(axis=0)

        ax.errorbar(
            np.round(np.arange(len(mean))*26.3/60, 2),
            mean,
            yerr=std,
            fmt='-o',
            capsize=3,
            color=colors[metric],
            label=metric_names[metric]
        )

    # Panel title
    ax.set_title(chosen_def.capitalize() if chosen_def != 'hemo' else 'HEMO', pad=10, fontsize=22)

    # Axis formatting
    ax.set_xticks(np.round(np.arange(len(mean))*26.3/60, 1))
    ax.grid(alpha=0.3)

    # Panel-specific y-limits
    if chosen_def == 'hemo':
        ax.set_ylim(0.09, 1.01)
    else:
        ax.set_ylim(0.49, 1.01)

# ------------------------
# Shared labels
# ------------------------
fig.supxlabel(
    # 'Number of Observed Measurements',
    'Timestamp from the Beginning of a Session (in hours)',
    y=0.02,
    fontsize=22
)
fig.supylabel(
    'AUC ROC Score',
    x=0.01,
    fontsize=22
)

# ------------------------
# Shared legend (top-center)
# ------------------------
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='upper center',
    ncol=4,
    frameon=True,
    bbox_to_anchor=(0.5, 1.08),
    fontsize=22
)

# ------------------------
# Final layout & export
# ------------------------
fig.tight_layout(rect=[0.03, 0.05, 1, 0.95])

plt.savefig(
    (FIGURE_ROOT + '/model_performance_over_time.pdf'),
    format="pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# 4. Resampling Resolution Sensitivity Result

In [ ]:
import matplotlib.pyplot as plt

def plot_clinical_bar(scores, title, remark=''):
    """
    Plots a publication-quality bar chart for clinical scores.
    
    Parameters:
    - scores: list of float, scores to plot
    - title: str, title of the plot
    """
    # X-axis labels
    time_labels = ['1min', '3mins', '5mins', '10mins', '15mins']
    
    # Figure setup
    plt.figure(figsize=(6, 4))

    cmap = plt.get_cmap('Blues')
    colors = [cmap(0.4 + 0.1*i) for i in range(len(scores))] 
    
    bars = plt.bar(time_labels, scores, color=colors, edgecolor='black', linewidth=0.9)
    
    # Annotate bars with values
    for bar, score in zip(bars, scores):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height + 0.015, f"{score:.3f}", 
                 ha='center', va='bottom', fontsize=16)
    
    # Style tweaks for "top-tier clinical journal" look
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.ylabel("AUC ROC", fontsize=16)
    plt.ylim(0.5, 0.96)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    
    # Remove top and right spines
    for spine in ['top', 'right']:
        plt.gca().spines[spine].set_visible(False)
    
    plt.tight_layout()

    if len(remark) > 0:
        plt.savefig((FIGURE_ROOT + '/{}.pdf').format(remark), format="pdf")
    plt.show()

# Example usage
scores = [0.717, 0.808, 0.849, 0.781, 0.772]
plot_clinical_bar(scores, "Fall 20, Different Resampling Rate", remark='fall20_diff_resample')


In [ ]:
scores = [0.883, 0.913, 0.944, 0.920, 0.926]
plot_clinical_bar(scores, "Nadir 90, Different Resampling Rate", remark='nadir90_diff_resample')


In [ ]:
scores = [0.697, 0.789, 0.814, 0.785, 0.773]
plot_clinical_bar(scores, "HEMO, Different Resampling Rate", remark='hemo_diff_resample')
